# T1 · Hop Count Audit
Subtasks: count the joins for eight questions in both schemas (by running them), self-checks, the written answer. All in this one notebook.

**Definition used literally:** one JOIN = one join operation beyond `FactShipment` itself. Snowflake = `DimCity → DimProvince → DimCountry`; everything else is unchanged.

In [1]:
import pandas as pd, os
pd.set_option("display.max_colwidth", None, "display.width", 250)

d = next(x for x in [".", "..", "/mnt/user-data/uploads"] if os.path.exists(os.path.join(x, "FactShipment.csv")))
rd = lambda n: pd.read_csv(os.path.join(d, n + ".csv"))
Fact, DimDate, DimRoute, DimMode, DimStatus = map(rd, ["FactShipment", "DimDate", "DimRoute", "DimMode", "DimStatus"])
DimCity_star = rd("DimCity")                                   # flat: Province, Region are columns
DimCity_snow, DimProvince, DimCountry = rd("DimCity_Snowflaked"), rd("DimProvince"), rd("DimCountry")

assert Fact.WeightKG.sum() == 578_650 and Fact.FreightCostPKR.sum() == 4_808_000 and len(Fact) == 34
# The flat DimCity has no Country column (every city is in Pakistan). For "by country" the star reads it
# from DimCity, so the star carries a Country attribute there:
DimCity_star = DimCity_star.assign(Country="Pakistan")

## The eight questions, executed in both schemas (every join is counted as it runs)

In [2]:
class J:
    """Counts every join operation beyond FactShipment."""
    def __init__(self): self.n, self.tables = 0, []
    def __call__(self, left, right, left_on, right_on, name):
        self.n += 1; self.tables.append(name)
        return left.merge(right, left_on=left_on, right_on=right_on, how="left", validate="m:1")

def run(q, schema):
    j = J(); f = Fact
    city = DimCity_star if schema == "star" else DimCity_snow
    def to_province(f, key):                                   # reach province from a city key
        f = j(f, city, key, "CityKey", "DimCity")
        if schema == "snow":
            f = j(f, DimProvince, "ProvinceKey", "ProvinceKey", "DimProvince")
            f = f.rename(columns={"ProvinceName": "Province"})
        return f
    if q == 1:
        f = j(f, DimDate, "DateKey", "DateKey", "DimDate");        out = f.groupby("MonthName").WeightKG.sum()
    elif q == 2:
        f = j(f, city, "OriginCityKey", "CityKey", "DimCity");     out = f.groupby("CityName").WeightKG.sum()
    elif q == 3:
        f = to_province(f, "OriginCityKey");                       out = f.groupby("Province").WeightKG.sum()
    elif q == 4:
        f = j(f, city, "OriginCityKey", "CityKey", "DimCity")
        if schema == "snow":
            f = j(f, DimProvince, "ProvinceKey", "ProvinceKey", "DimProvince")
            f = j(f, DimCountry, "CountryKey", "CountryKey", "DimCountry").rename(columns={"CountryName": "Country"})
        out = f.groupby("Country").WeightKG.sum()
    elif q == 5:
        f = to_province(f, "OriginCityKey"); f = j(f, DimDate, "DateKey", "DateKey", "DimDate")
        out = f.groupby(["Province", "MonthName"]).FreightCostPKR.sum()
    elif q == 6:
        f = j(f, DimMode, "ModeKey", "ModeKey", "DimMode"); f = j(f, DimStatus, "StatusKey", "StatusKey", "DimStatus")
        out = f.groupby(["ModeName", "StatusName"]).WeightKG.sum()
    elif q == 7:
        f = j(f, DimRoute, "RouteKey", "RouteKey", "DimRoute")
        g = f.groupby("Corridor"); out = g.FreightCostPKR.sum() / g.WeightKG.sum()      # ratio of two sums
    elif q == 8:
        f = to_province(f, "DestCityKey"); f = j(f, DimMode, "ModeKey", "ModeKey", "DimMode")
        out = f.groupby(["Province", "ModeName"]).WeightKG.sum()
    return j.n, " + ".join(j.tables), out

questions = ["Total weight by month", "Total weight by origin city", "Total weight by province", "Total weight by country",
             "Total cost by province, by month", "Total weight by mode and status", "Cost per kg by corridor",
             "Weight by destination province, by mode"]
why = {
 1: "Reads DimDate only; the city chain is not touched.",
 2: "Reads the city name, which lives on DimCity in both schemas; the split only moves Province and Region.",
 3: "Star: Province is a column on DimCity. Snowflake: it moved to DimProvince, so one extra hop.",
 4: "Star: country read off DimCity. Snowflake: DimCity to DimProvince to DimCountry, two extra hops.",
 5: "City side changes (province lives one hop further away); DimDate side does not.",
 6: "Reads DimMode and DimStatus only; the city chain is not touched.",
 7: "Corridor is an attribute of DimRoute; it never reads a city attribute, so no hop is added.",
 8: "Destination province is reached through DimCity, then DimProvince in the snowflake; DimMode is unchanged.",
}
rows, results = [], {}
for i, q in enumerate(questions, 1):
    ns, ts, rs = run(i, "star"); nk, tk, rk = run(i, "snow")
    results[i] = (rs, rk)
    rows.append((i, q, ts, tk, ns, nk, "Yes" if ns != nk else "No", why[i]))
t1 = pd.DataFrame(rows, columns=["#", "Business question", "Dimensions touched (star)", "Dimensions touched (snowflake)",
                                 "Star joins", "Snow joins", "Changed?", "Why it does or does not change"])
t1

,#,Business question,Dimensions touched (star),Dimensions touched (snowflake),Star joins,Snow joins,Changed?,Why it does or does not change
0,1,Total weight by month,DimDate,DimDate,1,1,No,Reads DimDate only; the city chain is not touched.
1,2,Total weight by origin city,DimCity,DimCity,1,1,No,"Reads the city name, which lives on DimCity in both schemas; the split only moves Province and Region."
2,3,Total weight by province,DimCity,DimCity + DimProvince,1,2,Yes,"Star: Province is a column on DimCity. Snowflake: it moved to DimProvince, so one extra hop."
3,4,Total weight by country,DimCity,DimCity + DimProvince + DimCountry,1,3,Yes,"Star: country read off DimCity. Snowflake: DimCity to DimProvince to DimCountry, two extra hops."
4,5,"Total cost by province, by month",DimCity + DimDate,DimCity + DimProvince + DimDate,2,3,Yes,City side changes (province lives one hop further away); DimDate side does not.
5,6,Total weight by mode and status,DimMode + DimStatus,DimMode + DimStatus,2,2,No,Reads DimMode and DimStatus only; the city chain is not touched.
6,7,Cost per kg by corridor,DimRoute,DimRoute,1,1,No,"Corridor is an attribute of DimRoute; it never reads a city attribute, so no hop is added."
7,8,"Weight by destination province, by mode",DimCity + DimMode,DimCity + DimProvince + DimMode,2,3,Yes,"Destination province is reached through DimCity, then DimProvince in the snowflake; DimMode is unchanged."


## Self-checks and the totals-must-not-move check

In [3]:
star_total, snow_total, changed = int(t1["Star joins"].sum()), int(t1["Snow joins"].sum()), int((t1["Changed?"] == "Yes").sum())
sc = pd.DataFrame([("Total star joins across the eight questions", 11, star_total),
                   ("Total snowflake joins",                     16, snow_total),
                   ("Questions whose join count changes",          4, changed)], columns=["Check", "Expected", "Yours"])
sc["Pass?"] = (sc.Expected == sc.Yours).map({True: "pass", False: "FIX"})
assert (sc["Pass?"] == "pass").all()

# Same answer in both schemas, and the totals did not move
for i, (rs, rk) in results.items():
    assert rs.sort_index().round(6).equals(rk.sort_index().round(6)), f"Q{i}: star and snowflake disagree"
assert all(results[i][0].sum() == 578_650 for i in (1, 2, 3, 4, 6, 8)) and results[5][0].sum() == 4_808_000
print("Star and snowflake return identical results for all 8 questions; weight 578,650 kg and cost PKR 4,808,000 hold in both.")
sc

Star and snowflake return identical results for all 8 questions; weight 578,650 kg and cost PKR 4,808,000 hold in both.


,Check,Expected,Yours,Pass?
0,Total star joins across the eight questions,11,11,pass
1,Total snowflake joins,16,16,pass
2,Questions whose join count changes,4,4,pass


## Evidence for the written answer: what the 5 extra joins actually join

In [4]:
extra = pd.DataFrame([
  ("DimCity -> DimProvince",     len(DimCity_snow), len(DimProvince), "questions 3, 4, 5, 8"),
  ("DimProvince -> DimCountry",  len(DimProvince),  len(DimCountry),  "question 4 only"),
], columns=["Extra hop", "Rows on the left", "Rows on the right", "Used by"])
print("Extra joins in total:", snow_total - star_total, "| rows in FactShipment:", len(Fact))
extra

Extra joins in total: 5 | rows in FactShipment: 34


,Extra hop,Rows on the left,Rows on the right,Used by
0,DimCity -> DimProvince,15,6,"questions 3, 4, 5, 8"
1,DimProvince -> DimCountry,6,2,question 4 only


## The written answer (4 marks)

**The eight questions need 5 more joins in total in the snowflake (16 against 11). That number counts join operations; it does not say which schema is faster, because it does not say what each join costs.**

- **Size of the tables joined.** All 5 extra joins are dimension-to-dimension lookups (15 city rows to 6 province rows, 6 province rows to 2 country rows), not another pass over the fact rows. A join against a 2-row table is not worth the same as a join against the fact.
- **Where the time goes.** In a columnar engine such as Power BI, the filter on `DimProvince` propagates through `DimCity` to the fact, and the dominant work is scanning and aggregating `FactShipment`; that scan is the same in both schemas, so 5 more joins does not mean 5 more units of work. At 34 rows neither schema is measurably slower.
- **What the count hides.** It says nothing about join order (small dimensions first), about the 4 questions that never read the city chain (identical cost in both), or about the trade-off on the other side: storage, maintenance, and how many relationships an analyst must follow. The star's real advantage is a shorter filter path (1 hop to a province, not 2 or 3), which is a simplicity argument measured by hop count, not a proven speed gain.